In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, vgg16_bn
from torchvision import transforms, models
from torch.utils.data import Subset
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ttest_rel
import time
import json
import os
from tqdm import tqdm
from utils import convert_label

In [12]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# settings
QUICK_EPOCHS = 20   # hyperparameter tuning's epoch number
FULL_EPOCHS = 100
CONFIGS = {
    'lr_0.001': {'lr': 0.001,'bs': 128,'wd': 5e-4},
    'lr_0.01': {'lr': 0.01,'bs': 128,'wd': 5e-4},
    'lr_0.1': {'lr': 0.1,'bs': 128,'wd': 5e-4},
    'bs_64': {'lr': 0.1,'bs': 64,'wd': 5e-4},
    'bs_256': {'lr': 0.1,'bs': 256,'wd': 5e-4},
    'wd_1e-3': {'lr': 0.1,'bs': 128,'wd': 1e-3},
    'wd_1e-5': {'lr': 0.1,'bs': 128,'wd': 1e-5},
}

torch.manual_seed(42)
np.random.seed(42)

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('plots', exist_ok=True)

# mean and standard deviation for normalizing/unnormalizing cifar-100 images
stats = ((0.507, 0.487, 0.441), (0.267, 0.256, 0.276))

# data augmentation and normalization
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                              transforms.ToTensor(), transforms.Normalize(*stats)])

test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=train_tf)
testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=test_tf)

# validation set from training data for tuning
n = len(trainset)
indices = torch.randperm(n).tolist()
tune_train = Subset(trainset, indices[:int(0.15*n)])
tune_val = Subset(trainset, indices[int(0.15*n):int(0.2*n)])

#print(f"train: {n}")
#print(f"tune: {len(tune_train)}")


100%|██████████| 169M/169M [00:10<00:00, 16.0MB/s]


In [11]:
# fast hyperparameter tuning to select best configuration( 20 epochs)
def tune(name, tune_train, tune_val):
    results = {}
    # loop over configs
    for cfg_name, cfg in CONFIGS.items():
        print(f"\n{cfg_name}: lr={cfg['lr']}, bs={cfg['bs']}, wd={cfg['wd']}")

        train_loader = torch.utils.data.DataLoader(tune_train, batch_size=cfg['bs'], shuffle=True, num_workers=2)
        val_loader = torch.utils.data.DataLoader(tune_val, batch_size=128, shuffle=False, num_workers=2)
        # model + optimizer + scheduler
        model = get_model(name)
        opt = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd']) # SGD+momentum optimizer
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, QUICK_EPOCHS)
        loss = nn.CrossEntropyLoss()

        best = 0
        for ep in range(QUICK_EPOCHS):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device),y.to(device)
                opt.zero_grad()
                loss(model(x),y).backward()
                opt.step()
            sched.step()

            if (ep+1)%5==0 or ep==QUICK_EPOCHS-1:   # doing validation in every 5 epochs
                model.eval()
                correct = 0

                for x,y in val_loader:
                    x, y = x.to(device),y.to(device)
                    outputs = model(x)              # predictions
                    preds = outputs.argmax(1)       # predicted class indices
                    correct += (preds == y).sum().item()

                acc = 100* correct / len(tune_val)
                best = max(best, acc)

        results[cfg_name] = {'cfg': cfg, 'val_acc': best}

    best_cfg = max(results, key=lambda k: results[k]['val_acc'])
    return results[best_cfg]['cfg'], results

# train, using best configuration
def train(model, name, cfg):
    loader = torch.utils.data.DataLoader(trainset, batch_size=cfg['bs'], shuffle=True, num_workers=2)
    opt = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd'])
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, FULL_EPOCHS)
    loss = nn.CrossEntropyLoss()

    accs = []
    losses = []
    for ep in range(FULL_EPOCHS):
        model.train()
        correct = total = epoch_loss = 0
        for x, y in loader:
            x, y = x.to(device),y.to(device)
            opt.zero_grad()
            outputs = model(x)              # predictions
            loss(outputs,y).backward()
            opt.step()
            # training accuracy
            preds = outputs.argmax(1)       # predicted class indices
            correct += (preds == y).sum().item()
            total += len(y)
            epoch_loss += loss(outputs,y).item()

        acc = 100* correct / total
        losses.append(epoch_loss / len(loader))
        accs.append(acc)
        sched.step()

    torch.save(model.state_dict(), f'models/{name}.pth')
    return {'train_accs': accs, 'train_losses': losses}


In [16]:
# Training on perturbed images to increase robustness against attacks
def train_adversarial(model, name, cfg, attack_type='pgd', eps=0.03, alpha=2/255, pgd_iters=7):
    loader = torch.utils.data.DataLoader(trainset, batch_size=cfg['bs'], shuffle=True, num_workers=2)
    opt = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['wd'])
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, FULL_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    accs = []
    losses = []

    for ep in range(FULL_EPOCHS):
        model.train()
        correct = 0
        total = 0
        epoch_loss = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # generating adversarial images
            model.eval()
            if attack_type == 'fgsm':
                attack_output = fgsm_attack(model, x, y, criterion, device, epsilon=eps)
            elif attack_type == 'pgd':
                attack_output = pgd_attack(model, x, y, criterion, device, epsilon=eps, alpha=alpha, iters=pgd_iters)
            else:
                attack_output = x #clean training

            if isinstance(attack_output, tuple):
                x_adv = attack_output[0] # to take just the image (bcz attack code returns both image and the noise/perturbation)
            else:
                x_adv = attack_output

            model.train()
            opt.zero_grad()
            outputs = model(x_adv)
            loss = criterion(outputs, y)
            loss.backward()
            opt.step()

            preds = outputs.argmax(1)
            correct += (preds==y).sum().item()
            total += len(y)
            epoch_loss += loss.item()

        acc = 100*correct/total
        losses.append(epoch_loss/ len(loader))
        accs.append(acc)
        sched.step()

    torch.save(model.state_dict(), f'models/{name}_adv_{attack_type}.pth')
    return {'train_accs': accs, 'train_losses': losses}

In [15]:
# resnet and vgg models
def get_model(name):
    if name == 'resnet':
        m = resnet18(weights=None)    # without pretrained ImageNet weights
        m.fc = nn.Linear(m.fc.in_features, 100)   # replacing final fc layer to output 100 classes for cifar-100
        m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False) # changing the first convolution a bit because cifar has low resolution
        m.maxpool = nn.Identity() # skip this early pooling step
    else:
        m = vgg16_bn(weights=None)
        m.classifier[6] = nn.Linear(4096, 100)
    return m.to(device)
# the .pth path should be changed according to the model
# they can be used if .pth files exist, meaning the training adversarial method should be ran once before
resnet = get_model('resnet')
vgg = get_model('vgg')
resnet.load_state_dict(torch.load("resnet_adv_pgd.pth", map_location="cuda:0"))
vgg.load_state_dict(torch.load("vgg_adv_pgd.pth", map_location="cuda:0"))

def fgsm_attack(model, images, labels, criterion = nn.CrossEntropyLoss(), device="cuda:0",epsilon = 0.01):
    model.to(device)
    images = images.to(device)
    labels = labels.to(device)
    images.requires_grad_(True)
    outputs = model(images)
    loss = criterion(outputs, labels).to(device)
    model.zero_grad()
    loss.backward()

    gradient = images.grad.data
    perturbations = epsilon * torch.sign(gradient)
    adversarial_images = images + perturbations
    #adversarial_images = torch.clamp(adversarial_images, 0, 1)

    return adversarial_images, perturbations

def get_adversarial_images(model, pil_images:list, labels:list, transform=None, unnormalize=None, pgd=True,
                           eps=0.03, alpha=1/255):
    if transform is None:
        transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            transforms.Normalize(mean=[0.507, 0.487, 0.441], std=[0.267, 0.256, 0.276])
        ])
        unnormalize = transforms.Compose([
            transforms.Normalize(mean=[0., 0., 0.], std=[1 / 0.267, 1 / 0.256, 1 / 0.276]),
            transforms.Normalize(mean=[-0.507, -0.487, -0.441], std=[1., 1., 1.]),
        ])
    image_batch = torch.stack([transform(image) for image in pil_images])
    batch_labels =  torch.tensor(labels)
    if pgd:
        adversarial_images, perturbations = pgd_attack(model, image_batch, batch_labels, epsilon=eps, alpha=alpha)
    else:
        adversarial_images, perturbations = fgsm_attack(model, image_batch, batch_labels, epsilon=eps)
    adversarial_images = [unnormalize(adversarial_image) for adversarial_image in adversarial_images]
    to_pil = transforms.ToPILImage()
    pil_adversarial_images = [to_pil(adversarial_image) for adversarial_image in adversarial_images]
    return pil_adversarial_images

def test_adversarial_images(original_images, adversarial_images, model, transform=None):
    model.eval()
    if transform is None:
        transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.507, 0.487, 0.441], std=[0.267, 0.256, 0.276])
        ])
    original_image_batch = torch.stack([transform(image) for image in original_images]).to("cuda:0")
    adversarial_image_batch = torch.stack([transform(image) for image in adversarial_images]).to("cuda:0")

    with torch.no_grad():
        orig_output = model(original_image_batch)
        adversarial_output = model(adversarial_image_batch)

    _, preds_orig = torch.max(orig_output, 1)
    _, preds_adv = torch.max(adversarial_output, 1)
    return preds_orig, preds_adv


def pgd_attack(model, images, labels, criterion = nn.CrossEntropyLoss(),
               device="cuda:0", epsilon=0.03, alpha=2 / 255, iters=40):

    orig_images = images.data
    model.to(device)
    images = images.to(device)
    labels = labels.to(device)
    orig_images = orig_images.to(device)

    for i in range(iters):
        images.requires_grad_(True)
        outputs = model(images)

        loss = criterion(outputs, labels).to(device)
        model.zero_grad()
        loss.backward()

        adv_images = images + alpha * images.grad.sign()
        eta = torch.clamp(adv_images - orig_images, min=-epsilon, max=epsilon)
        #images = torch.clamp(ori_images + eta, min=0, max=1).detach_()
        images = eta + orig_images
        images = images.detach_()

    return images, None

def test_cifar(ds, model_name, pgd, eps, alpha, batch_size = 256):
    if model_name == "resnet":
        model = get_model(model_name)
        model.load_state_dict(torch.load("resnet_adv_pgd.pth", map_location="cuda:0"))
    elif model_name == "vgg":
        model = get_model(model_name)
        model.load_state_dict(torch.load("vgg_adv_pgd.pth", map_location="cuda:0"))
    else:
        return

    pred_results = np.array([])
    pred_adv_results = np.array([])
    for i in range(0, len(ds), batch_size):
        batch_images = []
        batch_labels = []

        for j in range(i, min(i + batch_size, len(ds))):
            image = ds[j][0]
            label = ds[j][1]

            if image.mode == "RGB" and convert_label(label) is not None:

                batch_images.append(image)
                batch_labels.append(label)

        pil_adversarial_images = get_adversarial_images(model, batch_images, batch_labels,
                                                        pgd=pgd, eps=eps, alpha=alpha)
        preds_orig, preds_adv = test_adversarial_images(batch_images, pil_adversarial_images, model)
        preds_orig_np, preds_adv_np = preds_orig.cpu().numpy(), preds_adv.cpu().numpy()
        pred_adv_results = np.concatenate((pred_adv_results,preds_adv_np ))
        pred_results = np.concatenate((pred_results, preds_orig_np))
    num_changes = np.sum(np.not_equal(pred_adv_results, pred_results))
    print(f"[{model_name}] PGD:{pgd}, eps: {eps:.4f}, alpha: {alpha:.4f} => change: {100*num_changes/len(pred_results):.2f}%")


def get_images(ds, model_name, pgd, eps, alpha, indices):

    if model_name == "resnet":
        model = get_model(model_name)
        model.load_state_dict(torch.load("resnet_adv_pgd.pth", map_location="cuda:0"))
    elif model_name == "vgg":
        model = get_model(model_name)
        model.load_state_dict(torch.load("vgg_adv_pgd.pth", map_location="cuda:0"))
    else:
        return
    batch_images = []
    batch_labels = []
    for i in indices:
        image = ds[i][0]
        label = ds[i][1]
        if image.mode == "RGB" and convert_label(label) is not None:
            batch_images.append(image)
            batch_labels.append(label)
        if len(batch_images) == 5:
            break
    pil_adversarial_images = get_adversarial_images(model, batch_images, batch_labels,
                                                    pgd=pgd, eps=eps, alpha=alpha)

    for i, (img, adv_img) in enumerate(zip(batch_images,pil_adversarial_images)):
        adv_img.save(f"{model_name}\\img_{i}_PGD_{pgd}_eps_{eps:.4f}_alpha_{alpha:.4f}.png")
        img.save(f"{model_name}\\img_{i}.png")

ds = torchvision.datasets.CIFAR100(root='./datasets', train=False, download=True)

subset = [ds[i] for i in range(1000)]
'''
idx = [292,233,594,101,984, 12]

get_images(subset,model_name="resnet", pgd=True, eps=0.01, alpha=0.02, indices=idx)#
get_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.001, indices=idx)#
get_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.02, indices=idx)#
get_images(subset,model_name="resnet", pgd=False, eps=0.01, alpha=1/255, indices=idx)#
get_images(subset,model_name="resnet", pgd=False, eps=0.35, alpha=1/255, indices=idx)#

get_images(subset,model_name="vgg", pgd=True, eps=0.01, alpha=0.02, indices=idx)#
get_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.001, indices=idx)#
get_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.02, indices=idx)#
get_images(subset,model_name="vgg", pgd=False, eps=0.01, alpha=1/255, indices=idx)#
get_images(subset,model_name="vgg", pgd=False, eps=0.35, alpha=1/255, indices=idx)#

exit()
'''

test_cifar(subset,"resnet", pgd=True, eps=0.01, alpha=0.001)
test_cifar(subset,"resnet", pgd=True, eps=0.01, alpha=0.005)
test_cifar(subset,"resnet", pgd=True, eps=0.01, alpha=0.01)
test_cifar(subset,"resnet", pgd=True, eps=0.01, alpha=0.02)#
test_cifar(subset,"resnet", pgd=True, eps=0.05, alpha=0.001)
test_cifar(subset,"resnet", pgd=True, eps=0.05, alpha=0.005)
test_cifar(subset,"resnet", pgd=True, eps=0.05, alpha=0.01)
test_cifar(subset,"resnet", pgd=True, eps=0.05, alpha=0.02)
test_cifar(subset,"resnet", pgd=True, eps=0.1, alpha=0.001)
test_cifar(subset,"resnet", pgd=True, eps=0.1, alpha=0.005)
test_cifar(subset,"resnet", pgd=True, eps=0.1, alpha=0.01)
test_cifar(subset,"resnet", pgd=True, eps=0.1, alpha=0.02)
test_cifar(subset,"resnet", pgd=True, eps=0.2, alpha=0.001)#
test_cifar(subset,"resnet", pgd=True, eps=0.2, alpha=0.005)
test_cifar(subset,"resnet", pgd=True, eps=0.2, alpha=0.01)
test_cifar(subset,"resnet", pgd=True, eps=0.2, alpha=0.02)#

test_cifar(subset,"vgg", pgd=True, eps=0.01, alpha=0.001)
test_cifar(subset,"vgg", pgd=True, eps=0.01, alpha=0.005)
test_cifar(subset,"vgg", pgd=True, eps=0.01, alpha=0.01)
test_cifar(subset,"vgg", pgd=True, eps=0.01, alpha=0.02)#
test_cifar(subset,"vgg", pgd=True, eps=0.05, alpha=0.001)
test_cifar(subset,"vgg", pgd=True, eps=0.05, alpha=0.005)
test_cifar(subset,"vgg", pgd=True, eps=0.05, alpha=0.01)
test_cifar(subset,"vgg", pgd=True, eps=0.05, alpha=0.02)
test_cifar(subset,"vgg", pgd=True, eps=0.1, alpha=0.001)
test_cifar(subset,"vgg", pgd=True, eps=0.1, alpha=0.005)
test_cifar(subset,"vgg", pgd=True, eps=0.1, alpha=0.01)
test_cifar(subset,"vgg", pgd=True, eps=0.1, alpha=0.02)
test_cifar(subset,"vgg", pgd=True, eps=0.2, alpha=0.001)#
test_cifar(subset,"vgg", pgd=True, eps=0.2, alpha=0.005)
test_cifar(subset,"vgg", pgd=True, eps=0.2, alpha=0.01)
test_cifar(subset,"vgg", pgd=True, eps=0.2, alpha=0.02)#
'''
idx = [29,23,59,10,98]
test_cifar(subset, resnet, False, 0.4, 0,)
get_images(subset,model_name="resnet", pgd=True, eps=0.01, alpha=0.02, indices=idx)#
get_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.001, indices=idx)#
get_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.02, indices=idx)#
get_images(subset,model_name="resnet", pgd=False, eps=0.01, alpha=1/255, indices=idx)#
get_images(subset,model_name="resnet", pgd=False, eps=0.35, alpha=1/255, indices=idx)#

get_images(subset,model_name="vgg", pgd=True, eps=0.01, alpha=0.02, indices=idx)#
get_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.001, indices=idx)#
get_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.02, indices=idx)#
get_images(subset,model_name="vgg", pgd=False, eps=0.01, alpha=1/255, indices=idx)#
get_images(subset,model_name="vgg", pgd=False, eps=0.35, alpha=1/255, indices=idx)#

'''

[resnet] PGD:True, eps: 0.0100, alpha: 0.0010 => change: 8.13%
[resnet] PGD:True, eps: 0.0100, alpha: 0.0050 => change: 8.25%
[resnet] PGD:True, eps: 0.0100, alpha: 0.0100 => change: 8.01%
[resnet] PGD:True, eps: 0.0100, alpha: 0.0200 => change: 8.01%
[resnet] PGD:True, eps: 0.0500, alpha: 0.0010 => change: 32.78%
[resnet] PGD:True, eps: 0.0500, alpha: 0.0050 => change: 46.29%
[resnet] PGD:True, eps: 0.0500, alpha: 0.0100 => change: 47.01%
[resnet] PGD:True, eps: 0.0500, alpha: 0.0200 => change: 47.49%
[resnet] PGD:True, eps: 0.1000, alpha: 0.0010 => change: 32.89%
[resnet] PGD:True, eps: 0.1000, alpha: 0.0050 => change: 62.56%
[resnet] PGD:True, eps: 0.1000, alpha: 0.0100 => change: 64.71%
[resnet] PGD:True, eps: 0.1000, alpha: 0.0200 => change: 65.07%
[resnet] PGD:True, eps: 0.2000, alpha: 0.0010 => change: 32.89%
[resnet] PGD:True, eps: 0.2000, alpha: 0.0050 => change: 71.77%
[resnet] PGD:True, eps: 0.2000, alpha: 0.0100 => change: 75.72%
[resnet] PGD:True, eps: 0.2000, alpha: 0.020

'\nidx = [29,23,59,10,98]\ntest_cifar(subset, resnet, False, 0.4, 0,)\nget_images(subset,model_name="resnet", pgd=True, eps=0.01, alpha=0.02, indices=idx)#\nget_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.001, indices=idx)#\nget_images(subset,model_name="resnet", pgd=True, eps=0.2, alpha=0.02, indices=idx)#\nget_images(subset,model_name="resnet", pgd=False, eps=0.01, alpha=1/255, indices=idx)#\nget_images(subset,model_name="resnet", pgd=False, eps=0.35, alpha=1/255, indices=idx)#\n\nget_images(subset,model_name="vgg", pgd=True, eps=0.01, alpha=0.02, indices=idx)#\nget_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.001, indices=idx)#\nget_images(subset,model_name="vgg", pgd=True, eps=0.2, alpha=0.02, indices=idx)#\nget_images(subset,model_name="vgg", pgd=False, eps=0.01, alpha=1/255, indices=idx)#\nget_images(subset,model_name="vgg", pgd=False, eps=0.35, alpha=1/255, indices=idx)#\n\n'

In [9]:
# testing attacks
def test_fgsm(model, eps_list, testset, device):
    loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)
    loss = nn.CrossEntropyLoss()
    model.eval()
    accs = []
    preds_per_eps = []
    for eps in eps_list:
        correct_samples = []
        total = 0
        for x, y in loader:
            x, y = x.to(device),y.to(device)
            # FGSM attack
            adv = fgsm_attack(model, x, y, loss, device, epsilon=eps)
            correct_samples.extend((model(adv).argmax(1) == y).cpu().numpy())
            total += len(y)

        correct_samples = np.array(correct_samples)
        preds_per_eps.append(correct_samples)
        accs.append(100 * correct_samples.sum() / total)

    return accs, preds_per_eps

def test_pgd(model, eps, testset, device):
    loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)
    loss = nn.CrossEntropyLoss()
    model.eval()
    correct_samples = []
    total = 0
    for x, y in loader:
        x, y = x.to(device),y.to(device)
        # PGD attack
        adv = pgd_attack(model, x, y, loss, device, epsilon=eps)
        correct_samples.extend((model(adv).argmax(1) == y).cpu().numpy())
        total += len(y)

    correct_samples = np.array(correct_samples)
    acc = 100 * correct_samples.sum() / total

    return acc, correct_samples

In [ ]:
# tuning and training plots
def plot_results(r_tune, v_tune, r_hist, v_hist, r_fgsm, v_fgsm, r_clean, v_clean):

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    #parameter tuning comparison
    configs = list(r_tune.keys())
    r_vals = [r_tune[c]['val_acc'] for c in configs]
    v_vals = [v_tune[c]['val_acc'] for c in configs]

    x = np.arange(len(configs))
    width = 0.35

    axes[0, 0].bar(x - width/2, r_vals, width, label='ResNet18', alpha=0.8)
    axes[0, 0].bar(x + width/2, v_vals, width, label='VGG16', alpha=0.8)
    axes[0, 0].set_xlabel('Configuration')
    axes[0, 0].set_ylabel('Validation Accuracy (%)')
    axes[0, 0].set_title('Hyperparameter Tuning Results')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(configs, rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3)

    #training curves
    axes[0, 1].plot(r_hist['train_accs'], label='ResNet18', linewidth=2)
    axes[0, 1].plot(v_hist['train_accs'], label='VGG16', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Training Accuracy (%)')
    axes[0, 1].set_title('Training Progress')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    """
    #attack robustness
    axes[1, 0].plot(FGSM_EPS, r_fgsm, 'o-', label='ResNet18', linewidth=2, markersize=8)
    axes[1, 0].plot(FGSM_EPS, v_fgsm, 's-', label='VGG16', linewidth=2, markersize=8)
    axes[1, 0].axhline(y=r_clean, color='blue', linestyle='--', alpha=0.5, label='ResNet Clean')
    axes[1, 0].axhline(y=v_clean, color='orange', linestyle='--', alpha=0.5, label='VGG Clean')
    axes[1, 0].set_xlabel('FGSM Epsilon (ε)')
    axes[1, 0].set_ylabel('Accuracy (%)')
    axes[1, 0].set_title('FGSM Attack Robustness')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    """
    plt.tight_layout()
    plt.savefig('plots/analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# tuning
r_cfg, r_tune = tune('resnet', tune_train, tune_val)
v_cfg, v_tune = tune('vgg', tune_train, tune_val)

# training
resnet = get_model('resnet')
#r_hist = train(resnet, 'resnet', r_cfg) # run for clean train resnet
r_hist = train_adversarial(resnet, 'resnet', r_cfg, attack_type='pgd', eps=0.03)  # run for adv. train resnet (pgd attack)
#r_hist = train_adversarial(resnet, 'resnet', r_cfg, attack_type='fgsm', eps=0.03)  # run for adv. train resnet (fgsm attack)

vgg = get_model('vgg')
#v_hist = train(vgg, 'vgg', v_cfg)  # run for clean train vgg
v_hist = train_adversarial(vgg, 'vgg', v_cfg, attack_type='pgd', eps=0.03)  # run for adv. train vgg (pgd attack)
#v_hist = train_adversarial(vgg, 'vgg', v_cfg, attack_type='fgsm', eps=0.03)  # run for adv. train vgg (fgsm attack)

# clean test predictions
loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

resnet.eval()
correct_samples_resnet = []
with torch.no_grad():
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct_samples_resnet.extend((resnet(x).argmax(1) == y).cpu().numpy())

r_clean_pred =np.array(correct_samples_resnet)


vgg.eval()
correct_samples_vgg = []
with torch.no_grad():
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct_samples_vgg.extend((vgg(x).argmax(1) == y).cpu().numpy())

v_clean_pred = np.array(correct_samples_vgg)

r_clean = 100*r_clean_pred.sum()/len(testset)
v_clean = 100*v_clean_pred.sum()/len(testset)

print(f"\nClean Test Accuracy")
print(f"ResNet18: {r_clean:.2f}%")
print(f"VGG16: {v_clean:.2f}%")


In [ ]:
FGSM_EPS = [0, 0.01, 0.03, 0.05, 0.07, 0.1, 0.2, 0.3]
PGD_EPS = [0.3, 0.1]
# testing FGSM attacks
r_fgsm, r_fgsm_preds = test_fgsm(resnet, FGSM_EPS, testset, device)
v_fgsm, v_fgsm_preds = test_fgsm(vgg, FGSM_EPS, testset, device)

# testing PGD attacks
r_pgd, r_pgd_pred = test_pgd(resnet, PGD_EPS, testset, device)
v_pgd, v_pgd_pred = test_pgd(vgg, PGD_EPS, testset, device)

plot_results(r_tune, v_tune, r_hist, v_hist, r_fgsm, v_fgsm, r_clean, v_clean)
